# Step 2: Course Sequences with Pass/Fail 

tập trung vào course level với performance indication

Input: user.json, user-video.json (cho R_score)
Output: user_sequences.parquet (course-level only)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import json

spark = SparkSession.builder \
    .appName("Step2_CourseSequences") \
    .config("spark.driver.memory", "12g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

DATA_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data"
OUTPUT_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output"

MIN_COURSES_USER = 2
W_ENROLL = 0.3
W_WATCH = 0.7

print(f"✓ Spark {spark.version} ready")

✓ Spark 4.1.1 ready


## Step 1: Load Course List (with concepts - for reference only)

In [3]:
# Load course concepts (chỉ để lấy danh sách course hợp lệ)
concept_course = spark.read.csv(
    f"{DATA_PATH}/relations/concept-course_trans.csv", 
    header=True
)

valid_courses = concept_course.select("course_id").distinct()
valid_course_set = set([r.course_id for r in valid_courses.collect()])

print(f"Valid courses: {len(valid_course_set):,}")

Valid courses: 887


## Step 2: Calculate Watch Signal from User-Video

In [4]:
# Load user-video để tính engagement
user_video = spark.read.json(f"{DATA_PATH}/relations/user-video.json")
# Count videos watched per user
user_engagement = user_video.select(
    "user_id",
    size("seq").alias("videos_watched")
)

# Normalize to [0, 1] (cap at 100 videos)
user_engagement = user_engagement.withColumn(
    "watch_signal",
    least(col("videos_watched") / lit(100.0), lit(1.0))
)

user_engagement.cache()
print(f"✓ Users with video data: {user_engagement.count():,}")
user_engagement.show(5)

✓ Users with video data: 299,938
+-------+--------------+------------+
|user_id|videos_watched|watch_signal|
+-------+--------------+------------+
|  U_112|             6|        0.06|
|  U_150|             1|        0.01|
|  U_172|             1|        0.01|
|  U_189|            20|         0.2|
|  U_197|            55|        0.55|
+-------+--------------+------------+
only showing top 5 rows


## Step 3: Load User Enrollments

In [5]:
# Load users
user_df = spark.read.json(f"{DATA_PATH}/entities/user.json")

# Extract enrollments
user_enroll = user_df.select(
    col("id").alias("user_id"),
    posexplode("course_order").alias("pos", "course_id_raw"),
    col("enroll_time")
).withColumn("enroll_time_str", element_at(col("enroll_time"), col("pos") + 1))\
    .select(
        "user_id",
        col("pos"),
        concat(lit("C_"), col("course_id_raw").cast("string")).alias("course_id"),
        to_timestamp(col("enroll_time_str")).alias("enroll_time")
    ).filter(col("course_id").isin(valid_course_set))\
    .withColumn("enroll_signal", lit(1.0))

print(f"User enrollments: {user_enroll.count():,}")

User enrollments: 488,930


In [8]:
# Join watch signal
user_course = user_enroll.join(
    user_engagement,
    "user_id",
    "inner"
).withColumn(
    "watch_signal",
    coalesce(col("watch_signal"), lit(0.0))
)

# Tính R_score
user_course = user_course.withColumn(
    "R_score",
    W_ENROLL * col("enroll_signal") + W_WATCH * col("watch_signal")
)

# Normalize per user
user_window = Window.partitionBy("user_id")

# Pass/Fail indicator: R_score >= 0.4 = pass
user_course = user_course.withColumn(
    "passed",
    when(col("R_score") >= 0.4, lit(1)).otherwise(lit(0))
)

user_course.cache()
print(f"✓ User-courses: {user_course.count():,}")
user_course.show(10)

✓ User-courses: 77,582
+-------+---+---------+-------------------+-------------+--------------+------------+-------------------+------+
|user_id|pos|course_id|        enroll_time|enroll_signal|videos_watched|watch_signal|            R_score|passed|
+-------+---+---------+-------------------+-------------+--------------+------------+-------------------+------+
|  U_112|  0| C_696994|2019-09-20 16:34:04|          1.0|             6|        0.06|0.34199999999999997|     0|
|  U_112|  3| C_677020|2019-10-03 13:31:32|          1.0|             6|        0.06|0.34199999999999997|     0|
|  U_112|  8| C_696855|2020-01-22 09:23:39|          1.0|             6|        0.06|0.34199999999999997|     0|
|  U_112| 10| C_696679|2020-03-02 15:48:16|          1.0|             6|        0.06|0.34199999999999997|     0|
|  U_112| 12| C_697415|2020-03-28 12:54:07|          1.0|             6|        0.06|0.34199999999999997|     0|
|  U_112| 14| C_854866|2020-04-04 17:29:56|          1.0|             6| 

## Step 5: Filter Users with MIN_COURSES_USER

In [9]:
## Step 5: Filter Users with MIN_COURSES_USER

# Filter users
valid_users = user_course.groupBy("user_id").count().filter(col("count") >= MIN_COURSES_USER)

user_course = user_course.join(valid_users.select("user_id"), "user_id")

print(f"✓ Valid users: {valid_users.count():,}")
print(f"✓ Final user-courses: {user_course.count():,}")

# Thống kê pass/fail
print("\n--- Pass/Fail Statistics ---")
user_course.groupBy("passed").count().withColumn(
    "pct", col("count") / sum("count").over(Window.partitionBy())
).show()

✓ Valid users: 11,385
✓ Final user-courses: 71,479

--- Pass/Fail Statistics ---
+------+-----+-------------------+
|passed|count|                pct|
+------+-----+-------------------+
|     1|14769|0.20662012619090922|
|     0|56710| 0.7933798738090908|
+------+-----+-------------------+



## Step 6: Build Course Sequences

In [10]:
# Group sequences - CHỈ course level, không concepts
user_seq = user_course.withColumn(
    "sequence_order",
    col("pos") + 1
).groupBy("user_id").agg(
    sort_array(collect_list(struct(
        col("sequence_order"),
        col("course_id"),
        col("R_score"),
        col("passed")
    ))).alias("sequence")
)

user_seq.cache()
print(f"✓ Users with sequences: {user_seq.count():,}")


✓ Users with sequences: 11,385


## Step 7: Validation

In [11]:
# Sample
sample = user_seq.limit(5).collect()[0]
uid = sample["user_id"]
seq = sample["sequence"]

print(f"Sample User: {uid}")
print(f"Courses: {len(seq)}")
print("\nFirst 5 courses:")

for item in seq[:5]:
    status = "PASS" if item["passed"] else "FAIL"
    print(f"  {item['course_id']}: R_score={item['R_score']:.3f}, {status}")


Sample User: U_173698
Courses: 2

First 5 courses:
  C_696908: R_score=0.328, FAIL
  C_1756060: R_score=0.328, FAIL


## Step 8: Export

In [12]:
# Export
user_seq.write.mode("overwrite").json(f"{OUTPUT_PATH}/user_sequences_course_only")

user_seq.select(
    to_json(struct("user_id", "sequence")).alias("value")
).write\
    .mode("overwrite")\
    .text(f"{OUTPUT_PATH}/user_sequences_course_json")
user_seq.toPandas().to_json(f"{OUTPUT_PATH}/user_sequences.json", orient='records')

# Cleanup
user_engagement.unpersist()
user_course.unpersist()
user_seq.unpersist()

print("\n" + "="*50)
print("STEP 2 COMPLETE!")
print("Output: user_sequences_course_only")
print("="*50)


STEP 2 COMPLETE!
Output: user_sequences_course_only
